## Setup
- Install: `pip install -r requirements.txt`
- Data: 3 CSV files (customers, products, transactions)
- Goal: JOIN tables and uncover customer insights

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Database Detective - Case File #2026-003')

Database Detective - Case File #2026-003


## Part 1: Load the Three Tables
Load customers, products, and transactions separately.

In [ ]:
# Load all three CSV files
customers = pd.read_csv('data/customers.csv')
products = pd.read_csv('data/products.csv')
transactions = pd.read_csv('data/transactions.csv')

# Explore each table
print('Customers:', customers.shape)
print(customers.head())
print('\nProducts:', products.shape)
print(products.head())
print('\nTransactions:', transactions.shape)
print(transactions.head())

## Part 2: JOIN Transactions with Products
Merge transactions and products to see what was purchased.

**SQL equivalent:** `SELECT * FROM transactions JOIN products ON transactions.product_id = products.product_id`

In [7]:
# Merge transactions with products on product_id
tx_products = transactions.merge(products, on='product_id', how='left')

print(tx_products.head())
print('\nColumns:', tx_products.columns.tolist())
print('\nMissing product matches:', int(tx_products['name'].isna().sum()))

NameError: name 'transactions' is not defined

## Part 3: Calculate Revenue per Transaction
Add a revenue column: quantity * price

In [ ]:
# Create revenue column
tx_products['revenue'] = tx_products['quantity'] * tx_products['price']

print(tx_products[['transaction_id', 'quantity', 'price', 'revenue']].head())
print('\nRevenue summary:')
print(tx_products['revenue'].describe().round(2))

## Part 4: JOIN with Customers to Get Full Picture
Now add customer information.

**SQL equivalent:** `SELECT * FROM transactions JOIN products ... JOIN customers ON transactions.customer_id = customers.customer_id`

In [ ]:
# Merge with customers on customer_id
full_data = tx_products.merge(customers, on='customer_id', how='left')

print(full_data.head())
print('\nColumns:', full_data.columns.tolist())
print('\nMissing customer matches:', int(full_data['tier'].isna().sum()))

## Part 5: Top Customers by Total Spend
**SQL equivalent:** `SELECT customer_id, SUM(revenue) FROM ... GROUP BY customer_id ORDER BY SUM(revenue) DESC`

In [ ]:
# Group by customer and sum revenue
top_customers = (
    full_data.groupby(['customer_id', 'name'], as_index=False)['revenue']
    .sum()
    .sort_values('revenue', ascending=False)
    .reset_index(drop=True)
)

print('Top 10 Customers by Total Spend:')
print(top_customers.head(10))

## Part 6: Product Category Preferences by Customer Tier
Which tiers (Gold/Silver/Bronze) prefer which categories?

In [ ]:
# Group by tier and category, sum revenue
tier_category = (
    full_data.groupby(['tier', 'category'], as_index=False)['revenue']
    .sum()
    .sort_values(['tier', 'revenue'], ascending=[True, False])
)

tier_pivot = tier_category.pivot(index='tier', columns='category', values='revenue').fillna(0)
print(tier_pivot.round(2))

## Part 7: Churn Risk - Customers with No Recent Purchases
Find customers who haven't purchased in the last 30 days.

In [ ]:
# Convert transaction date to datetime
full_data['date'] = pd.to_datetime(full_data['date'])

# Get most recent purchase per customer
last_purchase = full_data.groupby('customer_id', as_index=False)['date'].max()
last_purchase.columns = ['customer_id', 'last_purchase_date']

# Identify customers with no purchase in last 30 days relative to case date
cutoff_date = pd.to_datetime('2026-01-27') - pd.Timedelta(days=30)
at_risk = last_purchase[last_purchase['last_purchase_date'] < cutoff_date].copy()

# Merge with customer names
at_risk = at_risk.merge(customers[['customer_id', 'name', 'tier']], on='customer_id', how='left')
at_risk = at_risk.sort_values('last_purchase_date')

print('Cutoff date:', cutoff_date.date())
print('Customers at churn risk (no purchase in 30 days):')
print(at_risk.head(20))
print('\nTotal at-risk customers:', len(at_risk))

## Part 8: Visualizations
Create bar charts for top customers and category preferences.

In [ ]:
# Bar chart - Top 10 customers by spend
top_10 = top_customers.head(10).sort_values('revenue', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_10['name'], top_10['revenue'], color='skyblue')
plt.title('Top 10 Customers by Total Spend')
plt.xlabel('Revenue ($)')
plt.ylabel('Customer')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar - Category revenue by tier
tier_pivot = tier_category.pivot(index='tier', columns='category', values='revenue').fillna(0)
tier_order = ['Gold', 'Silver', 'Bronze']
tier_pivot = tier_pivot.reindex([t for t in tier_order if t in tier_pivot.index])

ax = tier_pivot.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='tab20')
ax.set_title('Category Revenue by Customer Tier')
ax.set_xlabel('Tier')
ax.set_ylabel('Revenue ($)')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Part 9: Bonus - Product Associations (Optional)
Find which products are frequently bought together (same transaction or same customer).

In [ ]:
# Product association by customer basket (products bought by same customer)
from itertools import combinations
from collections import Counter

customer_products = (
    full_data.groupby('customer_id')['product_id']
    .apply(lambda s: sorted(set(s)))
    .reset_index(name='product_list')
)

pair_counter = Counter()
for product_list in customer_products['product_list']:
    if len(product_list) >= 2:
        pair_counter.update(combinations(product_list, 2))

pair_df = pd.DataFrame(
    [{'product_a': a, 'product_b': b, 'count_customers': c} for (a, b), c in pair_counter.items()]
).sort_values('count_customers', ascending=False)

# Attach readable product names
pair_df = pair_df.merge(products[['product_id', 'name']], left_on='product_a', right_on='product_id', how='left').drop(columns='product_id')
pair_df = pair_df.rename(columns={'name': 'product_a_name'})
pair_df = pair_df.merge(products[['product_id', 'name']], left_on='product_b', right_on='product_id', how='left').drop(columns='product_id')
pair_df = pair_df.rename(columns={'name': 'product_b_name'})

print('Top product pairings bought by same customers:')
print(pair_df[['product_a_name', 'product_b_name', 'count_customers']].head(10))

## Final Summary
- Top 5 customers: computed in Part 5 output (`top_customers.head(5)`).
- At-risk customers: computed in Part 7 output (`at_risk`).
- Category insights: shown in Part 6 pivot + Part 8 stacked chart.
- Recommendations for Jamie:
  1. Reward top spenders with tier-targeted loyalty offers to retain high-value customers.
  2. Launch win-back campaigns for at-risk customers before 30+ day inactivity grows.
  3. Bundle frequently co-purchased products from Part 9 to raise basket size.
  4. Personalize promotions by tier/category preference patterns from Part 6.